# Lecture 1 — IoT Sensor Data: Industrial Motor Monitoring

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ibrahimaldhaher/iot-sensor-health-monitor/blob/main/notebooks/01_sensor_monitoring.ipynb)

**Course:** Machine Learning and Deep Learning in IoT — MSc in Computer Science / IoT
**University of Sumer**, College of Computer Science and Information Technology, 2026–2027

---

### The exercise (Lecture 1, §1.22)

> Write a Python program that:
> 1. Stores the temperature and vibration measurements in two Python lists.
> 2. Calculates the average temperature and average vibration.
> 3. For each measurement, determines whether the machine is **Normal** or **Abnormal**:
>    `Temperature > 80 °C → Abnormal`, `Vibration > 5.0 → Abnormal`, otherwise `Normal`.
> 4. Prints the status of the machine for each measurement.
> 5. Prints the total number of abnormal measurements.
>
> ```python
> temperature = [72, 75, 83, 78, 85]
> vibration   = [2.1, 3.0, 6.2, 2.8, 5.5]
> ```

### What this notebook adds

| Part | Content |
|---|---|
| 1 | The literal answer — plain Python, no imports |
| 2 | The engineered version — typed, configurable, unit-tested package |
| 3 | Visual reading of the same result |
| 4 | From a hand-written rule to a *learned* rule (§1.4) — and where it should run (§1.16) |

## 0 · Setup

Runs unchanged in Colab and in VS Code.

In [ ]:
import pathlib
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/ibrahimaldhaher/iot-sensor-health-monitor.git"

if IN_COLAB:
    if not pathlib.Path("iot-sensor-health-monitor").exists():
        subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
    root = pathlib.Path("iot-sensor-health-monitor").resolve()
else:
    # Running from notebooks/ inside a local clone.
    root = pathlib.Path.cwd()
    if root.name == "notebooks":
        root = root.parent

sys.path.insert(0, str(root / "src"))
print("project root:", root)
print("environment :", "Google Colab" if IN_COLAB else "local / VS Code")

## 1 · The literal answer

Exactly what the question asks for: two lists, two averages, one rule, one counter.
No imports, no abstractions — this is the version that belongs on the answer sheet.

In [ ]:
temperature = [72, 75, 83, 78, 85]
vibration = [2.1, 3.0, 6.2, 2.8, 5.5]

TEMPERATURE_LIMIT = 80.0
VIBRATION_LIMIT = 5.0

average_temperature = sum(temperature) / len(temperature)
average_vibration = sum(vibration) / len(vibration)

print(f"Average temperature : {average_temperature:.2f} C")
print(f"Average vibration   : {average_vibration:.2f}")
print("-" * 52)

abnormal_count = 0
for i in range(len(temperature)):
    t, v = temperature[i], vibration[i]
    if t > TEMPERATURE_LIMIT or v > VIBRATION_LIMIT:
        status = "Abnormal"
        abnormal_count += 1
    else:
        status = "Normal"
    print(f"Measurement {i + 1}: T = {t:5.1f} C | V = {v:4.2f} -> {status}")

print("-" * 52)
print(f"Total abnormal measurements: {abnormal_count} out of {len(temperature)}")

## 2 · The engineered version

The same rule, written the way it would be written for a device that has to run it
every second for a year. Three things change, and each one is a deliberate decision:

- **The thresholds leave the comparison site.** `Thresholds` is an immutable object,
  so a different motor gets different limits without a code change — and the
  boundary behaviour becomes testable.
- **The verdict carries its reason.** `Abnormal` alone is not actionable; a
  maintenance team needs to know whether the motor is running *hot* or *shaking*.
- **The comparison is strictly greater than.** A reading sitting exactly on the
  limit stays Normal. That is the wording of the exercise, and it is also the
  classic off-by-one of threshold monitoring: get it wrong and every machine
  running at its rated maximum raises a false alarm.

In [ ]:
from iot_monitor import LECTURE_READINGS, Thresholds, evaluate_all, summarize
from iot_monitor.report import format_report

evaluations = evaluate_all(LECTURE_READINGS)
summary = summarize(evaluations)

print(format_report(evaluations, summary))

Because the limits are data rather than code, re-tuning the alarm is a one-line
experiment — the sensitivity question every condition-monitoring system eventually
has to answer:

In [ ]:
for limits in [Thresholds(), Thresholds(80.0, 4.0), Thresholds(76.0, 5.0)]:
    s = summarize(evaluate_all(LECTURE_READINGS, limits))
    print(
        f"T > {limits.temperature_c:>5.1f} C, V > {limits.vibration:>4.1f}"
        f"  ->  {s.abnormal_count}/{s.count} abnormal ({s.abnormal_rate:.0%})"
    )

## 3 · Reading the result visually

Temperature and vibration are measured on completely different scales, so they get
**one panel each** — never two y-axes on one plot, which would let the shape of the
chart be set by an arbitrary choice of scale rather than by the data.

Colour marks the breach, but it never carries the meaning alone: each bar that
crosses its limit is also labelled.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

SURFACE = "#fcfcfb"
INK = "#0b0b0b"
INK_MUTED = "#52514e"
GOOD = "#0ca30c"
CRITICAL = "#d03b3b"

indices = [e.reading.index for e in evaluations]
temps = [e.reading.temperature_c for e in evaluations]
vibs = [e.reading.vibration for e in evaluations]
overall = [e.status.value for e in evaluations]

panels = [
    ("Temperature", temps, 80.0, "°C", "{:.0f}"),
    ("Vibration", vibs, 5.0, "", "{:.1f}"),
]

fig, axes = plt.subplots(2, 1, figsize=(8.5, 6.4), sharex=True)
fig.patch.set_facecolor(SURFACE)

for ax, (name, values, limit, unit, fmt) in zip(axes, panels, strict=True):
    colors = [CRITICAL if v > limit else GOOD for v in values]
    bars = ax.bar(indices, values, width=0.55, color=colors, zorder=3)

    ax.axhline(limit, color=INK_MUTED, linestyle="--", linewidth=1.4, zorder=4)
    ax.text(
        0.52, limit, f"limit {fmt.format(limit)}{unit}",
        va="bottom", ha="left", fontsize=9, color=INK_MUTED, zorder=5,
        bbox={"facecolor": SURFACE, "edgecolor": "none", "pad": 1.5},
    )

    for bar, value in zip(bars, values, strict=True):
        if value > limit:
            ax.text(
                bar.get_x() + bar.get_width() / 2, value,
                f"{fmt.format(value)}{unit}  ▲",
                ha="center", va="bottom", fontsize=9.5,
                color=INK, fontweight="bold",
            )

    ax.set_title(f"{name} per reading", loc="left", fontsize=12,
                 fontweight="bold", color=INK, pad=10)
    ax.set_ylabel(f"{name} {unit}".strip(), fontsize=10, color=INK_MUTED)
    ax.set_ylim(0, max(max(values), limit) * 1.30)
    ax.set_xlim(0.45, len(indices) + 0.55)
    ax.set_facecolor(SURFACE)
    ax.grid(axis="y", color="#e4e3df", linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)
    for side in ("top", "right", "left"):
        ax.spines[side].set_visible(False)
    ax.spines["bottom"].set_color("#d8d7d2")
    ax.tick_params(colors=INK_MUTED, length=0, labelsize=10)

axes[1].set_xticks(indices)
axes[1].set_xticklabels(
    [f"{i}\n{s}" for i, s in zip(indices, overall, strict=True)], fontsize=9.5
)
axes[1].set_xlabel("Reading  ·  machine status", fontsize=10, color=INK_MUTED)

fig.suptitle(
    "Industrial motor — rule-based condition monitoring",
    x=0.055, y=0.985, ha="left", fontsize=14, fontweight="bold", color=INK,
)
fig.text(
    0.055, 0.938,
    f"{summary.abnormal_count} of {summary.count} readings abnormal"
    f"  ·  mean {summary.average_temperature_c:.1f} °C,"
    f" {summary.average_vibration:.2f} vibration",
    ha="left", fontsize=10.5, color=INK_MUTED,
)
fig.legend(
    handles=[
        Patch(facecolor=GOOD, label="within limit"),
        Patch(facecolor=CRITICAL, label="exceeds limit"),
    ],
    loc="upper left", bbox_to_anchor=(0.05, 0.905), ncol=2,
    frameon=False, fontsize=10, labelcolor=INK_MUTED,
)

fig.tight_layout(rect=[0, 0, 1, 0.875])
plt.show()

The plot says something the printed table does not: reading 3 breaks **both** limits
while reading 5 sits only just past each of them. Two readings are labelled the same
`Abnormal`, but they are not the same event — which is precisely the limitation of a
fixed rule, and the opening for §4.

## 4 · From a written rule to a learned one

§1.4 of the lecture draws the line:

```
Traditional programming :  Rules + Data            ->  Output
Machine learning        :  Data  + Desired Output  ->  Learned model
```

Everything above is the first line. The cell below is the second: instead of *writing*
`T > 80 or V > 5.0`, we hand a model labelled examples and let it find the boundary.

**Read the result honestly.** The labels here are generated from the rule itself plus
sensor noise, so a good score proves only that the model can *recover* a boundary we
already knew — not that it discovered anything. That is the right way to validate the
pipeline before real labelled failure data exists, and it is the setup the rest of the
course replaces with genuine historical data.

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree

rng = np.random.default_rng(seed=42)
N = 1200

T = rng.normal(loc=76, scale=6.0, size=N)
V = rng.normal(loc=3.6, scale=1.3, size=N).clip(0.1, None)

# Ground truth = the lecture rule, blurred by measurement noise near the limits,
# so the boundary is learnable but not trivially exact.
margin = rng.normal(0, 1.2, size=N)
y = (((T + margin) > 80) | ((V + margin * 0.09) > 5.0)).astype(int)

X = np.column_stack([T, V])
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

model = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Confusion matrix  [rows: actual, cols: predicted]")
print(confusion_matrix(y_test, y_pred), "\n")
print(
    classification_report(
        y_test, y_pred, target_names=["Normal", "Abnormal"], digits=3
    )
)

§1.19 warns that accuracy alone is misleading on imbalanced IoT data. The report above
is printed with **precision, recall and F1 per class** for that reason: on a fleet where
failures are rare, a model that predicts "Normal" forever scores high accuracy and
catches nothing.

The learned boundary can also be read directly — it should look like the rule we wrote:

In [ ]:
fig, ax = plt.subplots(figsize=(12.5, 5.4))
fig.patch.set_facecolor(SURFACE)
plot_tree(
    model,
    feature_names=["Temperature (°C)", "Vibration"],
    class_names=["Normal", "Abnormal"],
    filled=True, rounded=True, impurity=False, proportion=True,
    fontsize=8, ax=ax,
)
ax.set_title(
    "Learned decision tree — the splits land within ~1 unit of the written rule",
    loc="left", fontsize=12, fontweight="bold", color=INK, pad=12,
)
plt.show()

print("The five lecture readings, scored by the learned model:")
scores = model.predict(np.column_stack([temps, vibs]))
for e, p in zip(evaluations, scores, strict=True):
    learned = "Abnormal" if p else "Normal"
    flag = "" if learned == e.status.value else "   <- disagrees with the rule"
    print(
        f"  reading {e.reading.index}: T={e.reading.temperature_c:5.1f} "
        f"V={e.reading.vibration:4.2f} | rule={e.status.value:<8} "
        f"model={learned:<8}{flag}"
    )

## 5 · Where should this run? (§1.16)

The decision is small enough to run anywhere, and that is exactly what makes it a
useful example of the architectural question the course is built around.

| | Cloud inference | Edge inference |
|---|---|---|
| Latency | network round-trip per reading | microseconds, local |
| Bandwidth | every raw sample transmitted | only alarms and summaries |
| Availability | no link, no decision | keeps deciding offline |
| Compute / memory | effectively unlimited | a few KB on an MCU |
| Model updates | one place to retrain and deploy | fleet-wide rollout problem |
| Privacy | raw data leaves the site | raw data stays on the machine |

A decision tree of depth 3 is a handful of comparisons — it fits on the
microcontroller attached to the motor. So the honest architecture for *this* problem
is **edge inference, cloud training**: the motor decides in place, and only labelled
events travel upstream to improve the next model.

That is also the thread back to Topic 01 of the distributed-computing course: the
sensor is the new terminal, the cloud is the new host, and the controller is the first
escape from the centre.

---

### Summary

| Question | Answer |
|---|---|
| Average temperature | **78.60 °C** |
| Average vibration | **3.92** |
| Abnormal measurements | **2 of 5** — readings 3 (83 °C, 6.2) and 5 (85 °C, 5.5) |
| Which rule fired | reading 3: both · reading 5: both |